# **1. Import Libraries**

In [34]:
import pandas as pd
import numpy as np

# **2. Load Dataset**

In [37]:
df = pd.read_csv("IMDB Dataset.csv", engine='python')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [38]:
print("Shape of dataset:", df.shape)

Shape of dataset: (50000, 2)


# **3. Check Columns**

In [39]:
print(df.columns)

Index(['review', 'sentiment'], dtype='object')


In [40]:
print("\nSentiment Distribution:")
print(df['sentiment'].value_counts())


Sentiment Distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [41]:
print("\nMissing Values:")
print(df.isnull().sum())


Missing Values:
review       0
sentiment    0
dtype: int64


In [42]:
print("\nSample Review:")
print(df['review'][0])


Sample Review:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show i

# **4. NLP Preprocessing**

In [47]:
# Step 3: NLP Preprocessing

import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Download required resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

# Setup
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Cleaning function
def clean_text(text):
    text = str(text).lower()  # lowercase

    text = re.sub(r"http\S+|www\S+", "", text)  # remove URLs
    text = re.sub(r"<.*?>", "", text)  # remove HTML tags
    text = re.sub(r"@\w+", "", text)  # remove mentions
    text = re.sub(r"#", "", text)  # remove hashtags symbol

    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    text = re.sub(r'\d+', '', text)  # remove numbers

    tokens = word_tokenize(text)  # tokenization

    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words and len(word) > 2
    ]

    return " ".join(tokens)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


# **5. Apply Preprocessing**

In [48]:
# Apply preprocessing
df['clean_text'] = df['review'].apply(clean_text)

# Check result
df[['review', 'clean_text']].head()

,review,clean_text
0,One of the other reviewers has mentioned that ...,one reviewer mentioned watching episode youll ...
1,A wonderful little production. <br /><br />The...,wonderful little production filming technique ...
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,basically there family little boy jake think t...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter matteis love time money visually stunni...


# **6. Feature Engineering (Convert text → numbers)**

In [44]:
# Step 4: Feature Engineering (TF-IDF + Bag of Words)

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# 🔹 TF-IDF
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['clean_text'])

# 🔹 Bag of Words (BoW)
bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['clean_text'])

# 🔹 Convert target (sentiment → numeric)
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})
y = df['sentiment']

# Check shapes
print("TF-IDF Shape:", X_tfidf.shape)
print("BoW Shape:", X_bow.shape)

# Preview target
print("\nTarget Sample:")
print(y.head())

TF-IDF Shape: (50000, 5000)
BoW Shape: (50000, 5000)

Target Sample:
0    1
1    1
2    1
3    0
4    1
Name: sentiment, dtype: int64


# **7. Train Test Split**



In [45]:
# Step 5: Train-Test Split

from sklearn.model_selection import train_test_split

# 🔹 TF-IDF Split
X_train_tfidf, X_test_tfidf, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

# 🔹 BoW Split
X_train_bow, X_test_bow, y_train_bow, y_test_bow = train_test_split(
    X_bow, y, test_size=0.2, random_state=42
)

# Check shapes
print("TF-IDF Train Shape:", X_train_tfidf.shape)
print("TF-IDF Test Shape:", X_test_tfidf.shape)

print("\nBoW Train Shape:", X_train_bow.shape)
print("BoW Test Shape:", X_test_bow.shape)

TF-IDF Train Shape: (40000, 5000)
TF-IDF Test Shape: (10000, 5000)

BoW Train Shape: (40000, 5000)
BoW Test Shape: (10000, 5000)


# **8. Evaluate Models**

In [46]:
# Step 6: Model Training + Evaluation

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Evaluation function
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)

    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))


# =========================
# 🔹 Logistic Regression
# =========================
lr_tfidf = LogisticRegression(max_iter=200)
lr_tfidf.fit(X_train_tfidf, y_train)
evaluate_model("Logistic Regression (TF-IDF)", lr_tfidf, X_test_tfidf, y_test)

lr_bow = LogisticRegression(max_iter=200)
lr_bow.fit(X_train_bow, y_train_bow)
evaluate_model("Logistic Regression (BoW)", lr_bow, X_test_bow, y_test_bow)


# =========================
# 🔹 Naive Bayes
# =========================
nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train)
evaluate_model("Naive Bayes (TF-IDF)", nb_tfidf, X_test_tfidf, y_test)

nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, y_train_bow)
evaluate_model("Naive Bayes (BoW)", nb_bow, X_test_bow, y_test_bow)


# =========================
# 🔹 Decision Tree
# =========================
dt_tfidf = DecisionTreeClassifier()
dt_tfidf.fit(X_train_tfidf, y_train)
evaluate_model("Decision Tree (TF-IDF)", dt_tfidf, X_test_tfidf, y_test)

dt_bow = DecisionTreeClassifier()
dt_bow.fit(X_train_bow, y_train_bow)
evaluate_model("Decision Tree (BoW)", dt_bow, X_test_bow, y_test_bow)


Logistic Regression (TF-IDF)
Accuracy: 0.8865
Precision: 0.8782945736434109
Recall: 0.8993847985711451
F1 Score: 0.8887145798607706

Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.87      0.88      4961
           1       0.88      0.90      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000


Logistic Regression (BoW)
Accuracy: 0.8701
Precision: 0.8679653679653679
Recall: 0.8753720976384203
F1 Score: 0.8716529987155419

Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.86      0.87      4961
           1       0.87      0.88      0.87      5039

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000


Naive Bayes (TF-IDF)
Accuracy: 0

**Comparison + Insights + Conclusion**

**🔹 Model Comparison**
* Logistic Regression performed well because it handles high-dimensional text data efficiently.
* Naive Bayes was fast and gave decent results but slightly lower accuracy due to its independence assumption.
* Decision Tree showed higher accuracy in some cases but may overfit on text data.

**🔹 Feature Engineering Comparison**
* TF-IDF performed better because it gives importance to rare but meaningful words.
* Bag of Words (BoW) is simple and fast but does not consider word importance.

🔹 Key Observations
* Text preprocessing significantly improved model performance.
* Removing noise (punctuation, stopwords, HTML tags) helped in better feature extraction.
* TF-IDF is more effective for sentiment analysis than BoW.
* Logistic Regression is a strong baseline model for NLP tasks.
* Naive Bayes is efficient and works well for large datasets.
* Decision Tree can overfit when dealing with sparse text features.

**🔹 Final Conclusion**
* Best Model: Logistic Regression
* Best Feature Engineering: TF-IDF
* Proper preprocessing + feature engineering = better results

👉 Overall, the NLP pipeline successfully transformed raw text into meaningful features and built effective sentiment classification models.